<a href="https://colab.research.google.com/github/Lakshman3556/Deep_Learning/blob/main/DL_week11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implement Undercomplete AE and Overcomplete AE and write observations

An Autoencoder (AE) is a neural network used for unsupervised learning of efficient data representations. It consists of an encoder that compresses the input into a latent space and a decoder that reconstructs the original input. In an undercomplete autoencoder, the latent dimension is smaller than the input dimension, forcing the model to learn compressed representations. In contrast, an overcomplete autoencoder has a latent dimension greater than or equal to the input size, which may lead to identity mapping unless regularization is applied. Undercomplete AE is useful for dimensionality reduction, whereas overcomplete AE requires constraints to learn meaningful features.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
transform = transforms.ToTensor()
train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)

# AE Model
class AE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 512),
            nn.ReLU(),
            nn.Linear(512, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 3072),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out.view(-1, 3, 32, 32)

# Train function
def train(model):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for epoch in range(5):
        for x, _ in train_loader:
            x = x.to(device)
            out = model(x)
            loss = loss_fn(out, x)

            opt.zero_grad()
            loss.backward()
            opt.step()

        print(f"Epoch {epoch+1}, Loss: {loss.item()}")

# Undercomplete
under_model = AE(128)
train(under_model)

# Overcomplete
over_model = AE(4096)
train(over_model)

100%|██████████| 170M/170M [00:01<00:00, 85.5MB/s]


Epoch 1, Loss: 0.022049810737371445
Epoch 2, Loss: 0.014684176072478294
Epoch 3, Loss: 0.01157859805971384
Epoch 4, Loss: 0.01216720137745142
Epoch 5, Loss: 0.01082746684551239
Epoch 1, Loss: 0.020526058971881866
Epoch 2, Loss: 0.017722155898809433
Epoch 3, Loss: 0.01944396272301674
Epoch 4, Loss: 0.016997992992401123
Epoch 5, Loss: 0.01819559745490551


2. Implement Regularization in AE and demonstrate its use

In [ ]:
model = AE(4096).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.MSELoss()

for epoch in range(5):
    for x, _ in train_loader:
        x = x.to(device)
        out = model(x)

        loss = loss_fn(out, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.023055199533700943
Epoch 2, Loss: 0.024463742971420288
Epoch 3, Loss: 0.020642166957259178
Epoch 4, Loss: 0.019323628395795822


Implement Denoising AE and observe reconstruction

In [ ]:
def add_noise(x):
    noise = torch.randn_like(x) * 0.2
    return torch.clamp(x + noise, 0., 1.)

model = AE(128).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(5):
    for x, _ in train_loader:
        x = x.to(device)
        noisy = add_noise(x)

        out = model(noisy)
        loss = loss_fn(out, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")